# Model Comparison

imports and data loading

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/processed/clean_clothing_sales.csv")


In [2]:
#same steps as in notebook 02
cols_to_drop = [
    "brand",
    "product_category",
    "currency",
    "url",
    "product_id"
]

df = df.drop(columns=cols_to_drop)

# Create basic text features (part of clean data)
df['desc_len'] = df['description'].str.len().fillna(0)
df['name_len'] = df['name'].str.len().fillna(0)
df['name_word_count'] = df['name'].str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)

df['desc_word_count'] = df['description'].str.split().apply(lambda x: len(x) if isinstance(x, list) else 0)
df['price_per_word'] = df['price'] / (df['desc_word_count'] + 1) # +1 to avoid division by zero
df['price_per_name_word'] = df['price'] / (df['name_word_count'] + 1)

# Target transformation (for potential modeling, but saved with raw data)
df['sales_volume_log'] = np.log1p(df['sales_volume'])

# Text analysis - show example description lengths
df[['desc_len', 'desc_word_count']].describe()

# Define features and target (using new text features created in Notebook 01)
TARGET = 'sales_volume_log'
TEXT_FEATURES = ['description', 'name']
NUMERIC_FEATURES = [
    'price', 
    'desc_len', 
    'name_len', 
    'desc_word_count',
    'price_per_word',
    'name_word_count',
    'price_per_name_word'
    
]
CATEGORICAL_FEATURES = [
    'product_position',
    'section',
    'season',
    'material',
    'origin',
    'terms'
]

BINARY_FEATURES = [ 
    'promotion',
    'seasonal' 
]

for col in CATEGORICAL_FEATURES:
    df[col] = df[col].astype(str)


# Train-Test Split 
train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

# Define X/y matrices
all_features = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES + TEXT_FEATURES
X_train = train_df[all_features]
y_train = train_df[TARGET].values 
X_test = test_df[all_features]
y_test_log = test_df[TARGET].values # Save y_test as log-transformed
y_test_original = test_df['sales_volume'].values

# Text pipeline for description (TF-IDF)
tfidf = TfidfVectorizer(max_features=4000, ngram_range=(1,2), stop_words='english')
# Text vectorizer for name (TF-IDF)
tfidf_name = TfidfVectorizer(max_features=2000, ngram_range=(1,2), stop_words='english')

# Categorical pipeline: OneHotEncoding
cat_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Ensure all data in the column is converted to string
def to_string(X):
    return X.astype(str)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), NUMERIC_FEATURES),
    
    ('bin', 'passthrough', BINARY_FEATURES), 
    
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL_FEATURES),
   
    ('desc_txt', Pipeline([
        ('str_convert', FunctionTransformer(to_string, validate=False)),
        ('tfidf', TfidfVectorizer(max_features=4000, ngram_range=(1,2), stop_words='english'))
    ]), 'description'),
    
    ('name_txt', Pipeline([
        ('str_convert', FunctionTransformer(to_string, validate=False)),
        ('tfidf', TfidfVectorizer(max_features=2000, ngram_range=(1,2), stop_words='english'))
    ]), 'name')
], remainder='drop', verbose=False)

print("Preprocessor ready with StandardScaler, OneHotEncoder, and TfidfVectorizer.")


Preprocessor ready with StandardScaler, OneHotEncoder, and TfidfVectorizer.


# Multi-Model Comparison with Cross-Validation

In [3]:
# Define models to compare
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'KNeighbors': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    'SVR': SVR(kernel='rbf', C=100.0, gamma='scale'),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, tree_method='hist'),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

results = {}

print("Starting comprehensive multi-model evaluation (Cross-Validation + Test)...")

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    
    current_pipeline = Pipeline(steps=[
        ('pre', preprocessor),
        ('model', model)
    ])
    
    # 1. CROSS-VALIDATION STEP (Out-of-sample estimate on X_train)
    cv_scores = cross_val_score(
        current_pipeline, 
        X_train, 
        y_train, 
        cv=3, 
        scoring='neg_mean_absolute_error', 
        n_jobs=-1
    ) 
    
    mean_mae = -cv_scores.mean()
    std_mae = cv_scores.std()
    
    print(f"CV MAE Mean: {mean_mae:.4f} (±{std_mae:.4f})")
    results[name] = {'CV_MAE_Mean': mean_mae, 'CV_MAE_Std': std_mae}

    # 2. TRAIN-TEST EVALUATION STEP (Final fit for Test metrics)
    current_pipeline.fit(X_train, y_train)
    y_pred_log = current_pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_test_original, y_pred) # Use y_test_original
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred)) # Use y_test_original
    r2 = r2_score(y_test_original, y_pred) # Use y_test_original
    
    print(f"Test Metrics for {name}: MAE: {mae:.4f} | R2: {r2:.4f}")
    
    # Merge the Test results into the existing dictionary entry
    results[name].update({'MAE': mae, 'RMSE': rmse, 'R2': r2})


# Summarize results
print("\n==============================")
print("Model Comparison Summary (Sorted by CV MAE)")
print("==============================")
results_df = pd.DataFrame(results).T.sort_values(by='CV_MAE_Mean')
print(results_df)

# Store the best model name based on CV MAE
best_model_name = results_df.index[0]
print(f"\n--- Best Untuned Model: {best_model_name} ---")

Starting comprehensive multi-model evaluation (Cross-Validation + Test)...

--- Training LinearRegression ---
CV MAE Mean: 0.0589 (±0.0006)
Test Metrics for LinearRegression: MAE: 62.1396 | R2: 0.9282

--- Training DecisionTree ---
CV MAE Mean: 0.0758 (±0.0006)
Test Metrics for DecisionTree: MAE: 81.0943 | R2: 0.8779

--- Training KNeighbors ---
CV MAE Mean: nan (±nan)
Test Metrics for KNeighbors: MAE: 134.5842 | R2: 0.6461

--- Training SVR ---
CV MAE Mean: 0.0590 (±0.0008)
Test Metrics for SVR: MAE: 62.8174 | R2: 0.9247

--- Training RandomForest ---
CV MAE Mean: 0.0563 (±0.0005)
Test Metrics for RandomForest: MAE: 61.2399 | R2: 0.9307

--- Training GradientBoosting ---
CV MAE Mean: 0.0543 (±0.0006)
Test Metrics for GradientBoosting: MAE: 59.0018 | R2: 0.9353

--- Training XGBoost ---
CV MAE Mean: 0.0562 (±0.0006)
Test Metrics for XGBoost: MAE: 60.1013 | R2: 0.9329

--- Training LightGBM ---
CV MAE Mean: 0.0552 (±0.0007)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the o

c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


# Final Selection and Hyperparameter Tuning

In [4]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import numpy as np
from sklearn.metrics import mean_absolute_error, make_scorer

# 1. Define the Pipeline for the Best Model
# We use the model object from the 'models' dictionary defined earlier in the notebook.
best_model = models[best_model_name] # best_model_name = 'GradientBoosting'

tuning_pipeline = Pipeline(steps=[
    ('pre', preprocessor),
    ('model', best_model)
])

# 2. Define the Hyperparameter Space
# Parameters must be prefixed with 'model__'
param_distributions = {
    # Number of boosting stages/trees
    'model__n_estimators': randint(100, 500),
    # Learning rate shrinks the contribution of each tree (usually small)
    'model__learning_rate': uniform(0.01, 0.2), # From 0.01 up to 0.2
    # Maximum depth of the individual regression estimators
    'model__max_depth': randint(3, 8),
    # Minimum number of samples required to be at a leaf node
    'model__min_samples_leaf': randint(2, 20),
    # Subsample ratio of the training sample
    'model__subsample': uniform(0.6, 0.4), # From 0.6 up to 1.0 (0.6 + 0.4)
}

# Use MAE as the primary scoring metric to align with CV results
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

# 3. Setup and Run RandomizedSearchCV
# Use n_iter=20 for a reasonable search; increase this for production tuning
N_ITER = 20
CV_FOLDS = 3 

print(f"\nStarting RandomizedSearchCV for {best_model_name} (n_iter={N_ITER}, cv={CV_FOLDS})...")

random_search = RandomizedSearchCV(
    estimator=tuning_pipeline,
    param_distributions=param_distributions,
    n_iter=N_ITER,
    cv=CV_FOLDS,
    scoring=mae_scorer,
    random_state=42,
    verbose=1,
    n_jobs=-1 
)

random_search.fit(X_train, y_train)

# 4. Display Results and Finalize Pipeline
print("\n==================================")
print(f"Tuning Results for {best_model_name}")
print("==================================")
print("Best CV MAE Score:", -random_search.best_score_)
print("Best Parameters:", random_search.best_params_)

# Define and fit the FINAL pipeline using the best estimator found
final_best_pipeline = random_search.best_estimator_

# Final evaluation on the test set
y_pred_tuned_log = final_best_pipeline.predict(X_test)
y_pred_tuned_original = np.expm1(y_pred_tuned_log)

final_mae = mean_absolute_error(y_test_original, y_pred_tuned_original) 
final_r2 = r2_score(y_test_original, y_pred_tuned_original)

print(f"\n--- Final Tuned Model Test Metrics ---")
print(f"Test MAE (Tuned): {final_mae:.4f}") 
print(f"Test R2 (Tuned): {final_r2:.4f}")



Starting RandomizedSearchCV for GradientBoosting (n_iter=20, cv=3)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Tuning Results for GradientBoosting
Best CV MAE Score: 0.054563979825542015
Best Parameters: {'model__learning_rate': np.float64(0.013193250444042839), 'model__max_depth': 4, 'model__min_samples_leaf': 16, 'model__n_estimators': 363, 'model__subsample': np.float64(0.6137554084460873)}

--- Final Tuned Model Test Metrics ---
Test MAE (Tuned): 59.1131
Test R2 (Tuned): 0.9349


- Hyperparameter tuning did not substantially improve performance, implying the model was already near its optimal configuration.
The tuned model was kept for consistency.